<p align="center">
  <img src="https://upload.wikimedia.org/wikipedia/commons/thumb/8/8e/San_Francisco_Fire_Sacramento_Street_1906-04-18.jpg/1280px-San_Francisco_Fire_Sacramento_Street_1906-04-18.jpg" alt="Terremoto de San Francisco, 1906" width="700">
  <br>
  <em>Terremoto de San Francisco (18 de abril de 1906). Foto: Arnold Genthe, dominio público.</em>
</p>

# **Proyecto BBDD NoSQL: Dataset U.S Geological Survey**

## **1. Descripción de los datos**


El dataset contiene los **19.217 terremotos de magnitud 5 o superior** registrados en todo el mundo entre el 1 de enero de 2016 y el 24 de octubre de 2026. Se ha obtenido del catálogo **ComCat** del **U.S. Geological Survey (USGS)**, a través de su API pública ([FDSN Event Web Service](https://earthquake.usgs.gov/fdsnws/event/1/)), con una sola consulta:

```
https://earthquake.usgs.gov/fdsnws/event/1/query?format=geojson&starttime=2016-01-01&minmagnitude=5
```

El script `generar_dataset.py` descarga esta respuesta y la convierte en documentos para MongoDB, guardados en `datos/terremotos.json`. Los cambios principales son estos:
- Los campos se renombran al español.
- Las fechas pasan a tipo `Date`.
- La ubicación se guarda como un punto GeoJSON.
- Las listas de texto separadas por comas pasan a ser arrays.
- Los campos sin dato se omiten del documento.

Los datos se guardan en la base de datos `terremotos`, en la colección `terremotos`.

## Estructura de un documento

| Campo | Tipo | Descripción |
|---|---|---|
| `_id` | string | Identificador del evento en USGS |
| `magnitud`, `tipo_magnitud` | double, string | Magnitud (de 5,0 a 8,8) y escala usada |
| `lugar`, `region` | string | Descripción del lugar y país o zona (`region` se extrae de `lugar`) |
| `fecha` | date | Fecha y hora del terremoto (UTC) |
| `ubicacion` | GeoJSON Point | Epicentro `[longitud, latitud]` |
| `profundidad_km` | double | Profundidad del hipocentro |
| `tipo_evento` | string | `earthquake`, `volcanic eruption`, `nuclear explosion` o `landslide` |
| `tsunami` | bool | Si USGS emitió indicador de tsunami |
| `alerta` | string | Nivel de alerta por daños estimados: `green`, `yellow`, `orange` o `red` (solo en el 35 % de los documentos) |
| `significancia` | int | Índice de relevancia del evento según USGS |
| `impacto` | subdocumento | `personas_lo_sintieron`, `intensidad_reportada`, `intensidad_instrumental` (solo en el 57 %) |
| `calidad` | subdocumento | `num_estaciones`, `gap_azimutal`, `distancia_min_grados`, `rms` |
| `fuentes`, `productos` | array | Redes sísmicas que registraron el evento y productos científicos publicados (`shakemap`, `losspager`…) |



## **2. Cargar / importar el dataset**

Creamos la base de datos `terremotos` y la colección `terremotos` directamente desde Python con pymongo.



In [ ]:
from bson import json_util
from pymongo import MongoClient, GEOSPHERE

client = MongoClient("mongodb://localhost:27017", serverSelectionTimeoutMS=5000)
print("Servidor MongoDB", client.server_info()["version"])

Servidor MongoDB 9.0.2


### 1.1 Leer el fichero JSON
Usamos `bson.json_util` en lugar de `json` para que las fechas en formato `{"$date": ...}` se conviertan en objetos `datetime`
y se guarden en MongoDB como tipo `Date` (y no como texto).

In [12]:
with open("datos/terremotos.json", encoding="utf-8") as f:
    documentos = json_util.loads(f.read())

print(len(documentos), "documentos leídos")
print(type(documentos[0]["fecha"]))

19217 documentos leídos
<class 'datetime.datetime'>


### 1.2 Crear la base de datos y la colección e insertar los documentos
Si la colección ya existía la borramos antes, para que el notebook se pueda ejecutar varias veces sin errores de `_id` duplicado.

In [13]:
db = client["terremotos"]
db.drop_collection("terremotos")
col = db["terremotos"]

resultado = col.insert_many(documentos)
print(len(resultado.inserted_ids), "documentos insertados")
print("Bases de datos:", client.list_database_names())
print("Colecciones en 'terremotos':", db.list_collection_names())

19217 documentos insertados
Bases de datos: ['admin', 'clases', 'config', 'local', 'terremotos']
Colecciones en 'terremotos': ['terremotos']


### 1.3 Índices
Creamos índices sobre los campos por los que más vamos a filtrar u ordenar, y un índice geoespacial `2dsphere` sobre `ubicacion`.

In [4]:
col.create_index([("ubicacion", GEOSPHERE)])
col.create_index("fecha")
col.create_index("magnitud")
col.create_index("region")

for nombre, info in col.index_information().items():
    print(f"{nombre:15} {info['key']}")

_id_            [('_id', 1)]
ubicacion_2dsphere [('ubicacion', '2dsphere')]
fecha_1         [('fecha', 1)]
magnitud_1      [('magnitud', 1)]
region_1        [('region', 1)]


### 1.4 Comprobación

In [5]:
print("Documentos en la colección:", col.count_documents({}))
print("Primer terremoto:", col.find_one(sort=[("fecha", 1)])["fecha"])
print("Último terremoto:", col.find_one(sort=[("fecha", -1)])["fecha"])
col.find_one({"_id": "us6000qw60"})

Documentos en la colección: 19217
Primer terremoto: 2016-01-01 02:00:39.950000
Último terremoto: 2026-10-24 09:07:00.782000


{'_id': 'us6000qw60',
 'magnitud': 8.8,
 'tipo_magnitud': 'mww',
 'lugar': '2025 Kamchatka Peninsula, Russia Earthquake',
 'region': 'Russia',
 'fecha': datetime.datetime(2025, 7, 29, 23, 24, 52, 483000),
 'ubicacion': {'type': 'Point', 'coordinates': [160.2395, 52.4948]},
 'profundidad_km': 35,
 'tipo_evento': 'earthquake',
 'tsunami': True,
 'alerta': 'red',
 'significancia': 2114,
 'estado': 'reviewed',
 'red': 'us',
 'impacto': {'personas_lo_sintieron': 125,
  'intensidad_reportada': 9.1,
  'intensidad_instrumental': 8.732},
 'calidad': {'num_estaciones': 251,
  'gap_azimutal': 30,
  'distancia_min_grados': 1.102,
  'rms': 0.73},
 'fuentes': ['at', 'us', 'pt', 'usauto'],
 'productos': ['dyfi',
  'earthquake-name',
  'event-sequence',
  'finite-fault',
  'general-link',
  'general-text',
  'ground-failure',
  'impact-link',
  'impact-text',
  'internal-moment-tensor',
  'internal-origin',
  'losspager',
  'moment-tensor',
  'oaf',
  'origin',
  'phase-data',
  'shakemap'],
 'url': '